In [12]:
%pip install -q google-generativeai

Note: you may need to restart the kernel to use updated packages.


In [13]:
import pandas as pd
from google import genai
import os
from dotenv import load_dotenv
import tabulate
import io

# Nouveauté : Pydantic pour structurer la sortie
from pydantic import BaseModel, Field
from typing import List

# 1. Chargement des clés 
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")



# 3. Chargement des données
df = pd.read_csv("dirty_sales_data.csv")
print("Données chargées :", df.shape)

Données chargées : (530, 6)


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB


In [15]:
# buffer va permettre de transformer le .info en une string exploitable
buffer = io.StringIO()
df.info(buf=buffer)
info_str = buffer.getvalue()
print(info_str)
head_str = df.head(20).to_markdown()
print(head_str)


<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB

|    | Transaction_ID                       | Date       | Client_Name                   | Product_Category   | Unit_Price   |   Quantity |
|---:|:-------------------------------------|:-----------|:------------------------------|:-------------------|:-------------|-----------:|
|  0 | 07d1b815-eca3-4264-8cd8-1ec12bde4be2 | 2025-03-25 | De Oliveira Chartier S.A.R.L. | Clotting           | $389.54      |         14 |
|  1 | 17da96cd-5a59-4654-8feb-214c691c5698 | 2026-01-19 | Lebrun S.A.S.       

In [17]:
# Configuration de Gemini
Client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

prompt_v1 = f"""
Tu es un Data Analyst. Analyse ce dataset (info et head) et dis-loi ce qui ne va pas. 
Sois concis. 

DATA INFO : 
{info_str}

DATA HEAD :
{head_str}
"""
response_v1 = Client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt_v1
)

print(prompt_v1)


Tu es un Data Analyst. Analyse ce dataset (info et head) et dis-loi ce qui ne va pas. 
Sois concis. 

DATA INFO : 
<class 'pandas.DataFrame'>
RangeIndex: 530 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction_ID    530 non-null    str  
 1   Date              530 non-null    str  
 2   Client_Name       530 non-null    str  
 3   Product_Category  530 non-null    str  
 4   Unit_Price        530 non-null    str  
 5   Quantity          530 non-null    int64
dtypes: int64(1), str(5)
memory usage: 61.8 KB


DATA HEAD :
|    | Transaction_ID                       | Date       | Client_Name                   | Product_Category   | Unit_Price   |   Quantity |
|---:|:-------------------------------------|:-----------|:------------------------------|:-------------------|:-------------|-----------:|
|  0 | 07d1b815-eca3-4264-8cd8-1ec12bde4be2 | 2025-03-25 | De Oliveira Chartier S.A.R.L. |

In [ ]:
print(response_v1.text)

En tant que Data Analyst, voici le diagnostic de ce dataset : **il nécessite un nettoyage majeur (Data Cleaning) avant toute analyse.**

Voici ce qui ne va pas, colonne par colonne :

### 1. Types de données incorrects (Dtypes)
* **`Date`** : Stockée en `string` au lieu de `datetime`.
* **`Unit_Price`** : Stockée en `string` au lieu de `float`.

### 2. Problèmes de qualité de données
* **`Date`** :
  * **Formats incohérents** : Mélange de `YYYY-MM-DD` (ex: 2025-03-25) et `DD/MM/YYYY` (ex: 18/07/2025).
  * **Valeurs manquantes mal gérées** : Présence du texte `"TBD"` au lieu d'un `NaN`/`NaT`.
* **`Unit_Price`** :
  * **Formatage hétérogène** : Mélange de valeurs brutes (`338.46`), avec symbole (`$389.54`), avec devise (`793 USD`) et avec séparateurs de milliers (`$1,453.80`).
* **`Product_Category`** :
  * **Fautes de frappe (Coquilles)** : `Clotting` (au lieu de Clothing), `Electornics` (Electronics), `Hom` (Home). Cela va créer des doublons de catégories.
* **`Client_Name`** :
  * **C

In [ ]:
from google.genai import types

In [ ]:
# Définition de la structure attendue
from google.genai import types
import json

class DataIssue(BaseModel): 
    column : str = Field(..., description="Nom de la colonne concernée")
    issue_type : str = Field(..., description="Type de problème (ex: 'format', 'Duplicata', 'Valeur manquante')")
    description : str = Field(..., description="Explication courte du problème")
    suggested_action : str =Field(..., description="Action Python suggérée (ex: 'pd.todatetime', 'drop_duplicate')")

class DataAudit(BaseModel):
    issues : List[DataIssue]

# Appel à Gemini avec le schéma
prompt_v2 = f"""
Analyse ce dataset pour détecter les anomalies de qualité (type, doublons, formats). 
Remplis la structure JSON démandée.BaseModel

DATA INFO:
{info_str}

DATA HEAD:
{head_str}"""

response_v2 = Client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt_v2,
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",
        response_schema = DataAudit,
    ),
)

# Parsing automatique

audit_result = json.loads(response_v2.text)
print(audit_result)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
issues_list = audit_result["issues"]
# Affichage utilisable par le code 
pd.DataFrame(issues_list)

,column,issue_type,description,suggested_action
0,Date,format,"Formats de date multiples (YYYY-MM-DD, DD/MM/Y...","pd.to_datetime(df['Date'], errors='coerce')"
1,Client_Name,format,Incohérence dans la casse des noms de clients ...,df['Client_Name'].str.title()
2,Product_Category,format,Erreurs de saisie et fautes d'orthographe ('Cl...,df['Product_Category'].replace({'Clotting': 'C...
3,Unit_Price,type,Type chaîne de caractères contenant des symbol...,df['Unit_Price'] = df['Unit_Price'].str.replac...


In [ ]:
selected_actions = issues_list
print(f"Validation simulée de {len(selected_actions)} actions.")

selected_actions_restricted = selected_actions[0:4]
print(f"Validation simulée de {len(selected_actions_restricted)} actions.")

Validation simulée de 4 actions.
Validation simulée de 4 actions.


In [ ]:
selected_actions
selected_actions_restricted

[{'column': 'Date',
  'issue_type': 'format',
  'description': "Formats de date multiples (YYYY-MM-DD, DD/MM/YYYY) et présence de valeurs fictives comme 'TBD'",
  'suggested_action': "pd.to_datetime(df['Date'], errors='coerce')"},
 {'column': 'Client_Name',
  'issue_type': 'format',
  'description': 'Incohérence dans la casse des noms de clients (minuscules, majuscules)',
  'suggested_action': "df['Client_Name'].str.title()"},
 {'column': 'Product_Category',
  'issue_type': 'format',
  'description': "Erreurs de saisie et fautes d'orthographe ('Clotting', 'Electornics', 'Hom')",
  'suggested_action': "df['Product_Category'].replace({'Clotting': 'Clothing', 'Electornics': 'Electronics', 'Hom': 'Home'})"},
 {'column': 'Unit_Price',
  'issue_type': 'type',
  'description': "Type chaîne de caractères contenant des symboles monétaires ($, USD) et virgules au lieu d'un type numérique float",
  'suggested_action': "df['Unit_Price'] = df['Unit_Price'].str.replace(r'[\\$,\\sUSD]', '', regex=Tru

In [ ]:
# Définition de la structure de sortie avec Pydantic
class CodeOutput(BaseModel):
    python_script: str = Field(..., description="Le script Python complet et exécutable, sans blocs markdown.")

def generate_cleaning_code(df_sample, actions):
    cols_context = df_sample.dtypes.to_dict()

    prompt_code = f"""
    Tu es un développeur Python Sénior.
    Génères le code Python pour nettoyer le dataframe 'df' selon ces actions : 
    {json.dumps(actions, indent=2)}
    df : {df_sample.head(10)}
    Colonnes : {cols_context}

    Règles : 
    1. Code exécutable directement sur 'df'
    2. Gères les conversions avec robustesse.   
    """    

    # On force une sortie structurée contenant le code
    response = Client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt_code,
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",
        response_schema = CodeOutput,
    )
    )
    # On récupère proprement le script dans le JSON
    return json.loads(response.text)["python_script"]
cleaning_code = generate_cleaning_code(df, selected_actions)
print(cleaning_code)

NameError: name 'selected_actions' is not defined

In [ ]:
cleaning_code_restricted = generate_cleaning_code(df, selected_actions_restricted)
print(cleaning_code_restricted)

import pandas as pd

# 1. Convert Date to datetime handling mixed formats and coercing invalid values like 'TBD' to NaT
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed')

# 2. Standardize Client_Name case formatting
df['Client_Name'] = df['Client_Name'].str.title()

# 3. Correct spelling mistakes in Product_Category
category_replacements = {
    'Clotting': 'Clothing',
    'Electornics': 'Electronics',
    'Hom': 'Home'
}
df['Product_Category'] = df['Product_Category'].replace(category_replacements)

# 4. Clean Unit_Price by removing currency symbols, spaces, and converting to float
df['Unit_Price'] = df['Unit_Price'].astype(str).str.replace(r'[$,\sUSD]', '', regex=True)
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')


In [ ]:
print(cleaning_code)

import pandas as pd

# Convert Date column to datetime handling mixed formats and invalid values
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed')

# Standardize Client_Name formatting to Title Case
df['Client_Name'] = df['Client_Name'].str.title()

# Correct spelling mistakes in Product_Category
category_replacements = {
    'Clotting': 'Clothing',
    'Electornics': 'Electronics',
    'Hom': 'Home'
}
df['Product_Category'] = df['Product_Category'].replace(category_replacements)

# Clean currency symbols/text from Unit_Price and convert to float
df['Unit_Price'] = df['Unit_Price'].astype(str).str.replace('$', '', regex=False).str.replace('USD', '', regex=False).str.strip()
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')


In [ ]:
cleaning_code = cleaning_code.replace(
    "r'[$,|USD]'",
    "r'[$,|€USD]'"
)


In [ ]:
# Exécution sécurisée (Scope local)

df_clean_agent = df.copy()
local_scope = {"df": df_clean_agent, "pd": pd}

exec(cleaning_code, {}, local_scope)

df_final = local_scope["df"]
df_final.head()

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793.00,15


In [ ]:
cleaning_code_restricted = cleaning_code_restricted.replace(
     "r'[$,|USD]'",
     "r'[$,|€USD]'"
)


In [ ]:
# Exécution sécurisée (Scope local)

df_clean_agent_restricted = df.copy()
local_scope_restricted = {"df": df_clean_agent_restricted, "pd": pd}

exec(cleaning_code_restricted, {}, local_scope_restricted)

df_final_restricted = local_scope_restricted["df"]
df_final_restricted.head()

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,Software,338.46,19
3,69c0e851-82fe-4794-9a2e-d561db153ca2,2025-07-18,Pineau,Home,544.11,14
4,5bae6c36-9346-41ab-a62e-1b6e5e3aa67c,2025-10-31,Leroy,Electronics,793.00,15


In [ ]:
# === 1. ANALYSE ROBUSTE ===

# Prompt optimisé pour pousser l'IA dans ses rétranchements
prompt_robust_audit = f"""
Tu es un expert Data Engineer impitoyable sur la qualité de données. 
Audite ce dataset pour produire un plan de nettoyage en béton.

DATA HEAD:
{head_str}

DATA INFO:
{info_str}

Instructions de nettoyage :
1. STANDARDISATION : Unifie la casse des textes (ex: Noms en Title Case, Catégories en minuscule...)
2. NETTOAYE ROBUSTE : Pour les prix, utilise des Regex qui capturent tout (chiffres et séparateurs) et virent le reste.
3. LOGIQUE METIER : Détecte les anomalies sémantiques (Quantités négatives ? Doublons d'ID ?).
4. DATES : Sois capable de parser des formats mixtes. 

Retourne une liste strutured (JSON) des actions à mener. 
"""

print("🔍Analyse approfondie en cours...")
response_audit = Client.models.generate_content(
    model = "gemini-3.6-flash",
    contents = prompt_robust_audit,
    config = types.GenerateContentConfig(
        response_mime_type = "application/json",
        response_schema = DataAudit,
    ))
robust_issues = json.loads(response_audit.text)["issues"]
print(f"{len(robust_issues)} problèmes détectés.")

# === 2. GENERATION CODE
print("Génération du code de réparation...")
robust_code = generate_cleaning_code(df, robust_issues)
print("Code généré :")
print(robust_code)

# === EXECUTION ===
print(" Exécution...")
df_clean_final = df.copy()
scope = {"df": df_clean_final, "pd": pd}

try:
    exec(robust_code, {}, scope)
    df_final_result = scope["df"]
    print("🎉 Terminé ! Aperçu avant/après :")
    display(df.head(3))
    display(df_final_result.head(3))
    print(df_final_result.info())
except Exception as e:
    print(" Erreur exécution :", e)

🔍Analyse approfondie en cours...
6 problèmes détectés.
Génération du code de réparation...
Code généré :
import pandas as pd

# 1. Conversion de la date avec gestion des formats mixtes et erreurs
df['Date'] = pd.to_datetime(df['Date'], errors='coerce', format='mixed')

# 2. Normalisation du nom du client (Title Case)
df['Client_Name'] = df['Client_Name'].astype(str).str.title()

# 3. Correction des categories de produits et conversion en minuscules
category_corrections = {'Clotting': 'Clothing', 'Electornics': 'Electronics', 'Hom': 'Home'}
df['Product_Category'] = df['Product_Category'].replace(category_corrections).str.lower()

# 4. Nettoyage et conversion du prix unitaire en float
df['Unit_Price'] = df['Unit_Price'].astype(str).str.replace(',', '', regex=False).str.replace(r'[^0-9.]', '', regex=True)
df['Unit_Price'] = pd.to_numeric(df['Unit_Price'], errors='coerce')

# 5. Filtrage des valeurs de quantite invalides ou negatives
df = df[df['Quantity'] > 0]

# 6. Suppression des doublo

,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,Clotting,$389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,Clothing,$32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,TBD,Robert,Software,338.46,19


,Transaction_ID,Date,Client_Name,Product_Category,Unit_Price,Quantity
0,07d1b815-eca3-4264-8cd8-1ec12bde4be2,2025-03-25,De Oliveira Chartier S.A.R.L.,clothing,389.54,14
1,17da96cd-5a59-4654-8feb-214c691c5698,2026-01-19,Lebrun S.A.S.,clothing,32.14,8
2,a6008bd3-529c-4c19-a915-5239665e2db4,NaT,Robert,software,338.46,19


<class 'pandas.DataFrame'>
Index: 492 entries, 0 to 529
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Transaction_ID    492 non-null    str           
 1   Date              465 non-null    datetime64[us]
 2   Client_Name       492 non-null    str           
 3   Product_Category  492 non-null    str           
 4   Unit_Price        492 non-null    float64       
 5   Quantity          492 non-null    int64         
dtypes: datetime64[us](1), float64(1), int64(1), str(3)
memory usage: 53.3 KB
None
